# Ekman transport, MSE and Ekman advection maps

Plots one group (pentad or BSISO phase) of a composite NetCDF file written by `save_composite`. The library `ekman_V_ma_mo_plot.jl` holds the plot defaults, the reader, field computations, `panel` and `plot_panels`; the panels (data, labels, colors, limits) are set here. The library is loaded with Revise `includet`, so edits to it take effect here without restarting the kernel.

Kernel: "Julia", the juliaup default Julia, started with `--project=@.`, which finds `plot/Project.toml`.

In [1]:
using Revise
includet("ekman_V_ma_mo_plot.jl")

    CondaPkg Found dependencies: /home/ceoas/deszoeks/.julia/packages/CondaPkg/Pc7ZA/CondaPkg.toml
    CondaPkg Found dependencies: /home/ceoas/deszoeks/.julia/packages/PythonCall/trYos/CondaPkg.toml
    CondaPkg Found dependencies: /home/ceoas/deszoeks/.julia/packages/PythonPlot/96RVm/CondaPkg.toml
    CondaPkg Resolving changes
             - libstdcxx
             - libstdcxx-ng
    CondaPkg Initialising pixi
             │ /home/ceoas/deszoeks/.julia/artifacts/8429d049f6c01106c62971e9540e146dc068df25/bin/pixi
             │ init
             │ --format pixi
             └ /ceoas/deszoeks/projects/ekman-astz/plot/.CondaPkg
Created /ceoas/deszoeks/projects/ekman-astz/plot/.CondaPkg/pixi.toml
    CondaPkg Wrote /ceoas/deszoeks/projects/ekman-astz/plot/.CondaPkg/pixi.toml
             │ [dependencies]
             │ matplotlib = ">=1"
             │ openssl = ">=3, <3.6"
             │ 
             │     [dependencies.python]
             │     version = ">=3.10,!=3.14.0,!=3.14.1,<4"


"J kg\$^{-1}\$ km\$^{-1}\$"

## Data
For each pentad in `pentads`: `ds[i]`, its composite fields as read from the file, and `fs[i]`, the fields plotted (transport, MSE, gradients, advection, flux terms). Both are NamedTuples; `keys(fs[1])` lists them, e.g. `fs[1].adv_o`.

The composites use the uncurtailed Ekman transport. `ekman_fields` multiplies M by `r = curtail(lat, Ro_rad)` (`f.r`), a function of latitude only, so `Ro_rad` can be changed here without recompositing. It matters only within ~5° of the equator.

In [ ]:
file = "../ekman_pentad_clim_2012-2020.nc"
pentads = [28, 30, 32] # May 16-20, May 26-30, Jun 5-9 (pentad p: days 5p-4 to 5p of the 365-day year)
Ro_rad = 250e3 # m, curtail M by r = 1 - exp(-y^2/Ro_rad^2) near the equator (plotting only)
pvals = NCDataset(nc -> nc["pentad"][:], file) # pentads in the file
ds = [read_group(file, findfirst(==(p), pvals)) for p in pentads]
fs = [ekman_fields(d; Ro_rad) for d in ds]
title(p) = "$(basename(file)): pentad $p (Ro_rad = $(Ro_rad/1e3) km)"
keys(fs[1])

## Ekman transport, surface-air MSE and ocean-surface enthalpy
Each figure is a list of `panel(x, title, units; cmap="RdBu_r", kw...)`, filled in row by row (2 columns), written as a function of the fields `f` of one pentad; one figure is drawn per pentad. `x` is any lon x lat array, e.g. a field of `f`. `kw` go to `pcolormesh`, e.g. `vmin=-180, vmax=180`; without vmin/vmax the limits are ±98th percentile of |x|, and `vmin=nothing, vmax=nothing` lets matplotlib scale to the data. `figs1[i]` is `(fig, axs)` for pentad `pentads[i]`.

In [ ]:
panels1(f) = [
    panel(f.Mx, raw"Ekman transport $M_x$", raw"kg m$^{-1}$ s$^{-1}$"),
    panel(f.My, raw"Ekman transport $M_y$", raw"kg m$^{-1}$ s$^{-1}$"),
    panel(f.m_a ./ 1e3, raw"surface air MSE $m_a = c_{pa} T_2 + L_v q$", raw"kJ kg$^{-1}$"; cmap="viridis", vmin=nothing, vmax=nothing),
    panel(f.m_o ./ 1e3, raw"ocean surface enthalpy $m_o = c_{po}$ SST", raw"kJ kg$^{-1}$"; cmap="viridis", vmin=nothing, vmax=nothing)]
figs1 = [plot_panels(f.lon, f.lat, panels1(f); land=f.land, suptitle=title(p)) for (p, f) in zip(pentads, fs)];

In [ ]:
# figures display at the end of the cell that makes them; to adjust the axes, e.g. zoom in, and show them again:
# for (fig, axs) in figs1; axs[0, 0].set_xlim(40, 120); axs[0, 0].set_ylim(-20, 30); display(fig); end

## Gradients and Ekman advection
Columns: ocean, atmosphere; gradients per km. The advection is from the saved composites of daily products (`adv_*`); both advection panels share ±`advlim`.

In [ ]:
advlim = 180 # W m^-2
panels2(f) = [
    panel(1e3f.dmodx, raw"$\partial m_o/\partial x$", grad_units), panel(1e3f.dmadx, raw"$\partial m_a/\partial x$", grad_units),
    panel(1e3f.dmody, raw"$\partial m_o/\partial y$", grad_units), panel(1e3f.dmady, raw"$\partial m_a/\partial y$", grad_units),
    panel(f.adv_o, raw"ocean Ekman advection $-M_o \cdot \nabla m_o$", flux_units; vmin=-advlim, vmax=advlim),
    panel(f.adv_a, raw"atmosphere Ekman advection $-M_a \cdot \nabla m_a$, $M_a = -M_o$", flux_units; vmin=-advlim, vmax=advlim)]
figs2 = [plot_panels(f.lon, f.lat, panels2(f); land=f.land, suptitle=title(p)) for (p, f) in zip(pentads, fs)];

## Ocean and atmosphere Ekman advection and their sum, one pentad
Columns: ocean, atmosphere, sum. Rows: temperature (SST, $T_2$; mass-weighted, K kg m$^{-2}$ s$^{-1}$), and enthalpy (ocean $m_o$) and moist enthalpy (atmosphere $m_a$), W m$^{-2}$. Each row shares its color limits. Uses `advlim` from the cell above.

In [ ]:
p = 30 # May 26-30
f = fs[findfirst(==(p), pentads)]
Tlim = maximum(symlim, (f.advT_o, f.advT_a)) # shared by the temperature row
Hlim = advlim                                # W m^-2, shared by the enthalpy row
panels_adv = [
    panel(f.advT_o, raw"ocean $-M_o \cdot \nabla$SST", advT_units; vmin=-Tlim, vmax=Tlim),
    panel(f.advT_a, raw"atmosphere $-M_a \cdot \nabla T_2$", advT_units; vmin=-Tlim, vmax=Tlim),
    panel(f.advT_o .+ f.advT_a, "sum", advT_units; vmin=-Tlim, vmax=Tlim),
    panel(f.adv_o, raw"ocean enthalpy $-M_o \cdot \nabla m_o$, $m_o = c_{po}$SST", flux_units; vmin=-Hlim, vmax=Hlim),
    panel(f.adv_a, raw"atmosphere moist enthalpy $-M_a \cdot \nabla m_a$, $m_a = c_{pa}T_2 + L_v q$", flux_units; vmin=-Hlim, vmax=Hlim),
    panel(f.adv_o .+ f.adv_a, "sum", flux_units; vmin=-Hlim, vmax=Hlim)]
fig_adv, axs_adv = plot_panels(f.lon, f.lat, panels_adv; ncols=3, land=f.land, suptitle=title(p));

## Scalar times transport divergence and flux-form tendency
Columns: ocean, atmosphere. The flux form $-\nabla\cdot(M m)$ is from the saved composites of $s\tau_x$, $s\tau_y$; $m \nabla\cdot M = $ adv $-$ flux. m is measured from 0 K, so $m \nabla\cdot M$ is large and nearly cancels in the flux form.

In [ ]:
panels3(f) = [
    panel(f.sdiv_o, raw"ocean $m_o \nabla \cdot M_o$", flux_units),
    panel(f.sdiv_a, raw"atmosphere $m_a \nabla \cdot M_a$", flux_units),
    panel(f.flux_o, raw"ocean flux form $-\nabla \cdot (M_o m_o)$", flux_units),
    panel(f.flux_a, raw"atmosphere flux form $-\nabla \cdot (M_a m_a)$", flux_units)]
figs3 = [plot_panels(f.lon, f.lat, panels3(f); land=f.land, suptitle=title(p)) for (p, f) in zip(pentads, fs)];

## Save

In [ ]:
# for (p, (fig1, _), (fig2, _), (fig3, _)) in zip(pentads, figs1, figs2, figs3)
#     fig1.savefig("ekman_V_ma_mo_pentad$p.png"; dpi=150)
#     fig2.savefig("ekman_grad_adv_pentad$p.png"; dpi=150)
#     fig3.savefig("ekman_sdiv_flux_pentad$p.png"; dpi=150)
# end